# CFEVER：三階段事實查核作業

我把流程分成文件檢索、證據檢索與三分類。前兩階段使用凍結的中文 BERT 向量，第三階段另行 fine-tune 一個同款 BERT 分類器。

本作業以 CFEVER 訓練集標註過的頁面當作文件候選庫；在取回的頁面中，候選證據不限於訓練集標註句。這是縮小範圍的基線，不代表搜尋整個 Wikipedia。

程式與報告在 ChatGPT / OpenAI Codex 協助下改寫與解說；使用者需親自執行、檢查結果並完成最後的實測欄位。參考來源：[課程範例](https://github.com/IKMLab/NTHU_Natural_Language_Processing/tree/main/2026/Assignments/Assignment3)、[CFEVER](https://github.com/IKMLab/CFEVER-data)、[bert-base-chinese](https://huggingface.co/google-bert/bert-base-chinese)。


## 1. 套件與參數

建議在 Colab 選擇 GPU，先將競賽 ZIP 放進 Google Drive 的 `MyDrive/nlp-hw3/`。下方會掛載 Drive，將四個 JSONL 解壓到 Colab 暫存空間，並把模型和結果存回 Drive。

Kaggle 則請先將競賽資料加入 Input；首次下載模型需要 Internet，離線提交時請將完整模型目錄加入 Input 並設 `ALLOW_MODEL_DOWNLOAD = False`。詳細操作與執行時間注意事項見 `index.md`。


In [ ]:
# 生成式 AI 使用說明：ChatGPT / OpenAI Codex 協助改寫程式、檢查流程及撰寫說明。
# 參考：課程 sample_code.ipynb，以及 https://huggingface.co/google-bert/bert-base-chinese
import json
import random
import shutil
import urllib.request
import zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('使用裝置：', device)
if device.type == 'cpu':
    print('提醒：完整資料在 CPU 執行可能很久；建議切換 Colab / Kaggle GPU。')


使用裝置： cuda


In [ ]:
# 可改成自己的 Drive 資料夾；若放了多個 ZIP，請手動指定 COLAB_ZIP_PATH。
COLAB_DRIVE_DIR = Path('/content/drive/MyDrive/nlp-hw3')
COLAB_ZIP_PATH = None
required_files = ('train.jsonl', 'dev.jsonl', 'test.jsonl', 'wiki_pages.jsonl')

try:
    import google.colab  # 只有 Colab 才有這個套件。
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    DATA_DIR = Path('/content/cfever_data')
    OUTPUT_DIR = COLAB_DRIVE_DIR / 'output'
    MODEL_DIR = OUTPUT_DIR / 'bert-base-chinese'
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if not all((DATA_DIR / name).is_file() for name in required_files):
        zip_paths = [Path(COLAB_ZIP_PATH)] if COLAB_ZIP_PATH else sorted(COLAB_DRIVE_DIR.glob('*.zip'))
        if len(zip_paths) != 1:
            raise FileNotFoundError('請將一份競賽 ZIP 放入 MyDrive/nlp-hw3，或設定 COLAB_ZIP_PATH。')
        print('讀取競賽 ZIP：', zip_paths[0])
        with zipfile.ZipFile(zip_paths[0]) as archive:
            for filename in required_files:
                matches = [item for item in archive.infolist()
                           if not item.is_dir() and Path(item.filename).name == filename]
                if len(matches) != 1:
                    raise ValueError(f'ZIP 內必須有一份 {filename}；目前找到 {len(matches)} 份。')
                print('解壓：', filename)
                # 僅使用預先指定的檔名，避免直接解壓 ZIP 中任意路徑。
                temporary_path = DATA_DIR / (filename + '.part')
                with archive.open(matches[0]) as source, temporary_path.open('wb') as target:
                    shutil.copyfileobj(source, target)
                temporary_path.replace(DATA_DIR / filename)
elif Path('/kaggle/input').exists():
    DATA_DIR = Path('/kaggle/input')
    OUTPUT_DIR = Path('/kaggle/working')
    MODEL_DIR = None  # 如有多個模型，請直接填完整模型資料夾路徑。
else:
    DATA_DIR = Path('.')
    OUTPUT_DIR = Path('bert_student_output')
    MODEL_DIR = OUTPUT_DIR / 'bert-base-chinese'

def find_data_file(filename):
    direct = DATA_DIR / filename
    if direct.is_file():
        return direct
    matches = sorted(path for path in DATA_DIR.rglob(filename) if path.is_file())
    if len(matches) != 1:
        raise FileNotFoundError(f'{filename} 找到 {len(matches)} 份；請檢查 DATA_DIR：{DATA_DIR}')
    return matches[0]

TRAIN_PATH, DEV_PATH, TEST_PATH, WIKI_PATH = (find_data_file(name) for name in required_files)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, path in zip(required_files, (TRAIN_PATH, DEV_PATH, TEST_PATH, WIKI_PATH)):
    print(name, '→', path)
print('輸出目錄：', OUTPUT_DIR)

TRAIN_SAMPLES = 3000  # None 表示全部；先用小樣本確認流程，再考慮增加。
DEV_SAMPLES = 300     # 報告須寫出這次實際使用的筆數。
DOCUMENT_TOP_K = 5
EVIDENCE_TOP_K = 5
EMBEDDING_BATCH_SIZE = 16
RETRIEVAL_MAX_LENGTH = 256
EPOCHS = 1
BATCH_SIZE = 8
MAX_LENGTH = 192
LEARNING_RATE = 2e-5
ALLOW_MODEL_DOWNLOAD = True
if DOCUMENT_TOP_K < 1 or EVIDENCE_TOP_K < 1:
    raise ValueError('DOCUMENT_TOP_K 和 EVIDENCE_TOP_K 都必須至少為 1。')


Mounted at /content/drive
讀取競賽 ZIP： /content/drive/MyDrive/nlp-hw3/nthu-nlp-2026-assignment-3-fact-verification.zip
解壓： train.jsonl
解壓： dev.jsonl
解壓： test.jsonl
解壓： wiki_pages.jsonl
train.jsonl → /content/cfever_data/train.jsonl
dev.jsonl → /content/cfever_data/dev.jsonl
test.jsonl → /content/cfever_data/test.jsonl
wiki_pages.jsonl → /content/cfever_data/wiki_pages.jsonl
輸出目錄： /content/drive/MyDrive/nlp-hw3/output


### 讀取資料

JSONL 每一行是一筆 JSON。`label` 是標準標籤，`evidence` 是標準證據群組。
`all_train_df` 保留全部公開訓練資料，供建立檢索語料使用。
`train_df` 是本次用來訓練 BERT 的抽樣資料。
使用開啟的檔案物件讀取，避免把路徑字串誤判為 JSON 內容。



In [ ]:
with Path(TRAIN_PATH).open(encoding="utf-8-sig") as file:
    all_train_df = pd.read_json(file, lines=True)
train_df = all_train_df.copy()

with Path(DEV_PATH).open(encoding="utf-8-sig") as file:
    dev_df = pd.read_json(file, lines=True)

# 測試時只保留輸入欄位；模型不使用測試集標準答案。
with Path(TEST_PATH).open(encoding="utf-8-sig") as file:
    test_df = pd.read_json(file, lines=True)[["id", "claim"]]

if TRAIN_SAMPLES is not None:
    train_df = train_df.sample(n=min(TRAIN_SAMPLES, len(train_df)), random_state=SEED)
if DEV_SAMPLES is not None:
    dev_df = dev_df.sample(n=min(DEV_SAMPLES, len(dev_df)), random_state=SEED)

# 重設索引後，第 i 筆資料就能對應到第 i 個檢索結果。
train_df = train_df.reset_index(drop=True)
dev_df = dev_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("訓練／驗證／測試筆數：", len(train_df), len(dev_df), len(test_df))
train_df[["id", "claim", "label"]].head()



訓練／驗證／測試筆數： 3000 300 6000


,id,claim,label
0,26081,於1899年創設的步兵第26聯隊曾參與日俄戰爭，並廢止於1945年。,NOT ENOUGH INFO
1,11833,舊時稱作的打貓社現在是稱作中樂村，是嘉義縣的行政區。,supports
2,13777,從鬥牛運動演變而來的鬥牛舞是一種哈斯爾舞，男女動作都相當舒展、激烈。,refutes
3,1379,鳳凰是傳說的百鳥之王，與龍爲同等地位。,supports
4,7346,元朝王實甫創作雜劇《西廂記》。,supports


### 下載中文 BERT

先尋找 Input 中的完整 BERT 模型，找不到才考慮下載至工作目錄。
`/kaggle/working` 的互動執行檔案不會自動變成下一次提交的 Input。
`.part` 是尚未完成的下載檔；完整下載後才改成正式檔名。
如需離線重跑，請將模型資料夾以 Dataset 或 Notebook Output 加入 Input。



In [ ]:
MODEL_NAME = 'google-bert/bert-base-chinese'
MODEL_VERSION = '8f23c25b06e129b6c986331a13d8d025a92cf0ea'
model_files = ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'vocab.txt']

if MODEL_DIR is None:
    # 只自動選明確命名的模型資料夾，避免把其他 BERT 誤認成指定模型。
    model_folders = []
    for config_path in Path('/kaggle/input').rglob('config.json'):
        folder = config_path.parent
        if ('bert-base-chinese' in str(folder).lower()
                and all((folder / name).is_file() for name in model_files)):
            model_folders.append(folder)
    if len(model_folders) > 1:
        raise ValueError(f'找到多個模型，請在參數區指定 MODEL_DIR：{model_folders}')
    MODEL_DIR = model_folders[0] if model_folders else OUTPUT_DIR / 'bert-base-chinese'

MODEL_DIR = Path(MODEL_DIR)
missing_files = [name for name in model_files if not (MODEL_DIR / name).is_file()]
if missing_files:
    if str(MODEL_DIR).startswith('/kaggle/input/'):
        raise FileNotFoundError(f'Input 模型不完整，缺少：{missing_files}')
    if not ALLOW_MODEL_DOWNLOAD:
        raise FileNotFoundError('模型不完整；請加入完整模型或允許下載。')
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    for filename in missing_files:
        print('下載模型檔案：', filename)
        url = f'https://huggingface.co/{MODEL_NAME}/resolve/{MODEL_VERSION}/{filename}'
        temporary_path = MODEL_DIR / (filename + '.part')
        try:
            urllib.request.urlretrieve(url, temporary_path)
            temporary_path.replace(MODEL_DIR / filename)
        except OSError as error:
            raise RuntimeError('模型下載失敗；請確認 Internet，或先附加完整模型。') from error

model_config = json.loads((MODEL_DIR / 'config.json').read_text(encoding='utf-8'))
if model_config.get('model_type') != 'bert':
    raise ValueError('目前流程預期使用中文 BERT，請檢查 MODEL_DIR。')
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
print('模型目錄：', MODEL_DIR)
print('請確認 MODEL_DIR 指向的是', MODEL_NAME)


下載模型檔案： config.json
下載模型檔案： model.safetensors
下載模型檔案： tokenizer.json
下載模型檔案： tokenizer_config.json
下載模型檔案： vocab.txt
模型目錄： /content/drive/MyDrive/nlp-hw3/output/bert-base-chinese
請確認 MODEL_DIR 指向的是 google-bert/bert-base-chinese


## 2. Document Retrieval：找相關頁面

每筆標準證據是 `[annotation_id, evidence_id, page_title, sentence_id]`。只用公開訓練集的標註收集候選頁面，不使用 dev / test 的答案建立候選庫；再從 Wikipedia 逐行讀入這些頁面。

因此，若驗證集的正確頁面不曾出現在訓練集標註中，這個基線一定找不到。後面會印出驗證集的候選頁面涵蓋率，幫助看出這個上限。


In [ ]:
def gold_groups(evidence):
    """只回傳有效的非空證據組，避免空集合被誤判為完整命中。"""
    groups = []
    if not isinstance(evidence, (list, tuple)):
        return groups
    for group in evidence:
        if not isinstance(group, (list, tuple)) or not group:
            continue
        pairs = []
        for item in group:
            if (not isinstance(item, (list, tuple)) or len(item) < 4
                    or not isinstance(item[2], str) or not item[2]
                    or not isinstance(item[3], int) or item[3] < 0):
                pairs = []
                break
            pairs.append((item[2], item[3]))
        if pairs:
            groups.append(set(pairs))
    return groups

wanted_pages = set()
for row in all_train_df.itertuples():
    if str(row.label).upper() != 'NOT ENOUGH INFO':
        for group in gold_groups(row.evidence):
            wanted_pages.update(title for title, _ in group)

wiki_pages = {}
with WIKI_PATH.open(encoding='utf-8') as file:
    for line in file:
        page = json.loads(line)
        if page['id'] in wanted_pages:
            wiki_pages[page['id']] = page
if not wiki_pages:
    raise ValueError('找不到訓練集標註過的 Wikipedia 頁面，請檢查資料。')
print('訓練標註中的頁面數：', len(wanted_pages))
print('實際載入的候選頁面數：', len(wiki_pages))


訓練標註中的頁面數： 3188
實際載入的候選頁面數： 3188


### 用 BERT 將文字轉成向量

BERT 為每個 token 產生一個向量，這裡用 **mean pooling** 將它們平均成文字向量。
平均時使用 attention mask 排除 padding，避免補齊長度的 token 影響結果。
再用 **L2 normalization** 把向量長度變成 1，這樣內積就等於 cosine similarity。

頁面與 claim 使用同一個預訓練 BERT，這個檢索模型不參與後面的分類器訓練。
每頁只編碼標題與開頭文字，超過 `RETRIEVAL_MAX_LENGTH` 的 token 會被截斷。
同一個 encode_texts 函式會用於頁面、claim 和證據句的編碼。
原始 BERT 未針對檢索做對比學習訓練，這裡用來示範向量檢索。



In [ ]:
retrieval_model = AutoModel.from_pretrained(str(MODEL_DIR), local_files_only=True).to(device)
retrieval_model.eval()
retrieval_model.requires_grad_(False)
retrieval_parameters = sum(parameter.numel() for parameter in retrieval_model.parameters())
print('檢索模型參數量：', retrieval_parameters)
if retrieval_parameters >= 1_000_000_000:
    raise ValueError('檢索模型必須小於 1B 參數。')

def encode_texts(texts):
    """批次輸入文字，回傳已 L2 正規化、儲存在 CPU 的向量。"""
    if not texts:
        raise ValueError('沒有可供編碼的文字，請檢查輸入資料。')
    embeddings = []
    with torch.no_grad():
        for start in range(0, len(texts), EMBEDDING_BATCH_SIZE):
            batch_texts = texts[start:start + EMBEDDING_BATCH_SIZE]
            inputs = tokenizer(batch_texts, padding=True, truncation=True,
                               max_length=RETRIEVAL_MAX_LENGTH, return_tensors='pt')
            inputs = {name: value.to(device) for name, value in inputs.items()}
            token_vectors = retrieval_model(**inputs).last_hidden_state
            mask = inputs['attention_mask'].unsqueeze(-1).float()
            vector_sum = (token_vectors * mask).sum(dim=1)
            token_count = mask.sum(dim=1).clamp(min=1)
            text_vectors = F.normalize(vector_sum / token_count, p=2, dim=1)
            embeddings.append(text_vectors.cpu())
            if (start // EMBEDDING_BATCH_SIZE + 1) % 200 == 0:
                print(f'已編碼 {min(start + len(batch_texts), len(texts))}/{len(texts)} 筆')
    return torch.cat(embeddings, dim=0)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: /content/drive/MyDrive/nlp-hw3/output/bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


檢索模型參數量： 102267648


### 建立頁面向量索引

每個頁面只需要編碼一次，之後所有 claim 都與這些固定向量比較。
此處的索引就是一個矩陣，每一列對應 `page_titles` 中的一個頁面。
BERT-base 的向量維度為 768，索引保留在 CPU，以節省 GPU 記憶體。



In [ ]:
page_titles = sorted(wiki_pages)
document_texts = []
for title in page_titles:
    text = title + " " + wiki_pages[title]["text"][:6000]
    document_texts.append(text)

document_embeddings = encode_texts(document_texts)
print("頁面向量矩陣大小：", document_embeddings.shape)



已編碼 3188/3188 筆
頁面向量矩陣大小： torch.Size([3188, 768])


### 用 cosine similarity 找最相關的頁面

把 claim 也轉成單位向量，再計算 `claim_embeddings @ document_embeddings.T`。
結果矩陣的每一列是一個 claim，每一欄是一個頁面；數值越大，向量方向越接近。
`topk` 選出每個 claim 相似度最高的頁面。Cosine similarity 可為負數，仍依排名取前 K 個。
claim 向量只需計算一次，下一階段選句時會重用。



In [ ]:
def retrieve_documents(claim_embeddings, top_k, query_batch_size=256):
    """分批計算相似度，避免一次建立過大的 claim × 文件矩陣。"""
    results = []
    count = min(top_k, len(page_titles))
    for start in range(0, len(claim_embeddings), query_batch_size):
        vectors = claim_embeddings[start:start + query_batch_size]
        similarities = vectors @ document_embeddings.T
        for indices in similarities.topk(k=count, dim=1).indices.tolist():
            results.append([page_titles[index] for index in indices])
    return results

train_claim_embeddings = encode_texts(train_df['claim'].tolist())
dev_claim_embeddings = encode_texts(dev_df['claim'].tolist())
test_claim_embeddings = encode_texts(test_df['claim'].tolist())
train_documents = retrieve_documents(train_claim_embeddings, DOCUMENT_TOP_K)
dev_documents = retrieve_documents(dev_claim_embeddings, DOCUMENT_TOP_K)
test_documents = retrieve_documents(test_claim_embeddings, DOCUMENT_TOP_K)
print('測試 claim：', test_df.loc[0, 'claim'])
print('取回頁面：', test_documents[0])


已編碼 3200/6000 筆
測試 claim： 沈懋孝爲明朝的平民，對書籍厭惡，毀壞數量達萬卷。
取回頁面： ['明憲宗', '侯景之亂', '王莽', '蘇軾', '嚴郢']


### Document Retrieval Recall@N

只統計 SUPPORTS / REFUTES，若前 N 頁包含任一組標準證據所需的所有頁面，該題算命中。未找到有效標準證據組的題目仍列入分母、算未命中，並額外印出筆數。


In [ ]:
def is_verifiable(row):
    return str(row.label).upper() in {'SUPPORTS', 'REFUTES'}

def document_group_hit(groups, predicted_pages):
    predicted_pages = set(predicted_pages)
    return any({title for title, _ in group}.issubset(predicted_pages) for group in groups)

document_hits = document_count = coverage_hits = missing_gold = 0
for index, row in enumerate(dev_df.itertuples()):
    if not is_verifiable(row):
        continue
    document_count += 1
    groups = gold_groups(row.evidence)
    missing_gold += not bool(groups)
    document_hits += document_group_hit(groups, dev_documents[index][:DOCUMENT_TOP_K])
    coverage_hits += document_group_hit(groups, wiki_pages)

if document_count == 0:
    raise ValueError('驗證樣本沒有 SUPPORTS / REFUTES，無法計算檢索 Recall。')
document_recall = document_hits / document_count
document_coverage = coverage_hits / document_count
print(f'Document Retrieval Recall@{DOCUMENT_TOP_K}: {document_recall:.4f} ({document_hits}/{document_count})')
print(f'候選頁面涵蓋率：{document_coverage:.4f} ({coverage_hits}/{document_count})')
print('沒有有效證據組的可驗證題數：', missing_gold)


Document Retrieval Recall@5: 0.0149 (3/201)
候選頁面涵蓋率：0.0149 (3/201)
沒有有效證據組的可驗證題數： 0


## 3. Evidence Retrieval：從頁面挑選句子

這次只處理至少一個 train / dev / test claim 實際取回的頁面，但會將這些頁面的所有非空句子納入候選，不限定為訓練集標註過的句子。這裡使用 test 的 claim 做正常檢索，沒有讀取 test 的標準答案。

句子必須保留 Wikipedia 原始編號；使用相同的 BERT、mean pooling、L2 normalization 和 cosine similarity 排序。句子多時這一步可能花較久，可觀察編碼進度。


In [ ]:
selected_pages = {title for results in (train_documents, dev_documents, test_documents)
                  for pages in results for title in pages}
sentence_info = []
sentence_texts = []
page_to_sentence_indices = defaultdict(list)
for title in sorted(selected_pages):
    for line in wiki_pages[title]['lines'].splitlines():
        parts = line.split('\t')
        if len(parts) < 2 or not parts[0].isdigit():
            continue
        sentence_id, text = int(parts[0]), parts[1].strip()
        if text:
            page_to_sentence_indices[title].append(len(sentence_info))
            sentence_info.append((title, sentence_id))
            sentence_texts.append(text)
if not sentence_texts:
    raise ValueError('取回的頁面沒有可用句子，請檢查 Wikipedia 資料格式。')
print('實際需要編碼的頁面數：', len(selected_pages))
print('候選證據句數：', len(sentence_info))
sentence_embeddings = encode_texts(sentence_texts)
print('句子向量矩陣大小：', sentence_embeddings.shape)

# 三種文字都已編碼；先釋放檢索模型的 GPU 記憶體，再訓練分類器。
retrieval_model.to('cpu')
del retrieval_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


實際需要編碼的頁面數： 2927
候選證據句數： 20374
已編碼 3200/20374 筆
已編碼 6400/20374 筆
已編碼 9600/20374 筆
已編碼 12800/20374 筆
已編碼 16000/20374 筆
已編碼 19200/20374 筆
句子向量矩陣大小： torch.Size([20374, 768])


In [ ]:
def retrieve_evidence(claim_embeddings, document_results, top_k):
    """只從取回的頁面選句，回傳原始座標和分類器用的文字。"""
    all_evidence, all_contexts = [], []
    for claim_vector, pages in zip(claim_embeddings, document_results):
        candidate_indices = [index for title in pages
                             for index in page_to_sentence_indices.get(title, [])]
        evidence, selected_texts = [], []
        if candidate_indices:
            scores = sentence_embeddings[candidate_indices] @ claim_vector
            count = min(top_k, len(candidate_indices))
            for local_index in scores.topk(k=count).indices.tolist():
                index = candidate_indices[local_index]
                title, sentence_id = sentence_info[index]
                evidence.append([title, sentence_id])
                selected_texts.append(sentence_texts[index])
        all_evidence.append(evidence)
        all_contexts.append(' '.join(selected_texts)[:1000])
    return all_evidence, all_contexts

train_evidence, train_contexts = retrieve_evidence(train_claim_embeddings, train_documents, EVIDENCE_TOP_K)
dev_evidence, dev_contexts = retrieve_evidence(dev_claim_embeddings, dev_documents, EVIDENCE_TOP_K)
test_evidence, test_contexts = retrieve_evidence(test_claim_embeddings, test_documents, EVIDENCE_TOP_K)
print('證據座標：', test_evidence[0])
print('證據文字：', test_contexts[0])


證據座標： [['明憲宗', 1], ['蘇軾', 6], ['嚴郢', 16], ['王莽', 5], ['侯景之亂', 1]]
證據文字： 明憲宗在位二十三年 ， 期間恢復其叔朱祁鈺的帝號 ， 又爲于謙等忠臣平反 ， 初年勵精圖治 ， 體恤民情 ， 任用李賢 、 商輅 、 彭時等賢臣 ， 頗爲時人所傳誦 ； 在軍事方面 ， 整飭戎政 ， 對內平定荊襄群盜和西南傜蠻 ， 對外抵禦韃靼女真 、 收復河套 、 經略哈密 ， 立下不少功績 。 在王安石變法期間 ， 蘇軾雖贊同政治應該改革 ， 但反對操之過急的政策及王安石任用的後任呂惠卿 ， 因此招來新黨爪牙李定橫加陷害 ； 後來又因蘇軾反對元祐更化盡廢新法 ， 受到司馬光爲首的舊黨斥退 ， 終生當不了宰相 。 嚴郢擔任京兆尹時 ， 生性峻暴 ， “ 敢誅殺 ， 盜賊一衰 ， 減隸官匠丁數百千人 ， 號稱職尹 。 ” 他終其一生都自奉甚儉 、 治家嚴苛 ， 曾先後命三名行爲不端的嫡子自盡 。 蕭梁都城建康 （ 今南京城 ） 被圍城時爆發嚴重饑荒和食人事件 ， 屍體更再引發瘟疫 ， 城破前建康人口十萬 ， 城破後不足四千 （ < 4 % ） 。


### Evidence Retrieval Recall@N

只統計 SUPPORTS / REFUTES，檢查前 N 句是否完整涵蓋任一組標準證據；頁面名稱與原始句號都要相同。此處評估實際的兩階段流程，不假設前一階段已找到正確頁面。


In [ ]:
def evidence_group_hit(groups, predicted_evidence):
    predicted = {tuple(pair) for pair in predicted_evidence}
    return any(group.issubset(predicted) for group in groups)

evidence_hits = 0
for index, row in enumerate(dev_df.itertuples()):
    if is_verifiable(row):
        evidence_hits += evidence_group_hit(gold_groups(row.evidence), dev_evidence[index][:EVIDENCE_TOP_K])
evidence_recall = evidence_hits / document_count
print(f'Evidence Retrieval Recall@{EVIDENCE_TOP_K}: {evidence_recall:.4f} ({evidence_hits}/{document_count})')


Evidence Retrieval Recall@5: 0.0149 (3/201)


## 4. Classification：用 BERT 預測標籤

將 claim 與證據作為一組文字配對：`[CLS] claim [SEP] context [SEP]`。
tokenizer 把文字轉為 token ID、attention mask 等模型輸入。
模型的三個輸出分別對應 supports、refutes、NOT ENOUGH INFO。



In [ ]:
labels = ["supports", "refutes", "NOT ENOUGH INFO"]
label_to_id = {"SUPPORTS": 0, "REFUTES": 1, "NOT ENOUGH INFO": 2}
train_labels = train_df["label"].str.upper().map(label_to_id).tolist()

class ClaimDataset(Dataset):
    def __init__(self, claims, contexts, target_labels=None):
        # 固定長度 padding，讓 DataLoader 可以直接把多筆資料堆成一個 batch。
        self.encodings = tokenizer(
            list(claims), contexts, truncation=True,
            padding="max_length", max_length=MAX_LENGTH,
        )
        self.labels = target_labels

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, index):
        item = {}
        for name, values in self.encodings.items():
            item[name] = torch.tensor(values[index], dtype=torch.long)
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[index], dtype=torch.long)
        return item

train_dataset = ClaimDataset(train_df["claim"], train_contexts, train_labels)
dev_dataset = ClaimDataset(dev_df["claim"], dev_contexts)
test_dataset = ClaimDataset(test_df["claim"], test_contexts)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)



### 模型與 Optimizer

`AutoModelForSequenceClassification` 會在預訓練 BERT 上加入三分類層。
分類層是新初始化的，因此載入時看到相關提示是正常的，需要用下面的資料訓練它。
傳入 `labels` 時，模型會自動計算 Cross Entropy loss。



In [ ]:
torch.manual_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR), num_labels=3, local_files_only=True,
    id2label=dict(enumerate(labels)),
    label2id={label: index for index, label in enumerate(labels)},
).to(device)
classification_parameters = sum(parameter.numel() for parameter in model.parameters())
print('分類模型參數量：', classification_parameters)
if classification_parameters >= 1_000_000_000:
    raise ValueError('分類模型必須小於 1B 參數。')
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/nlp-hw3/output/bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your d

分類模型參數量： 102269955


### Training

每批資料依序執行：清空梯度、計算預測與 loss、反向傳播、更新參數。
`model.train()` 會啟用訓練模式；gradient clipping 用來限制過大的梯度。



In [ ]:
training_losses = []
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for step, batch in enumerate(train_loader, start=1):
        # 將這個 batch 的每個 tensor 搬到 GPU（或 CPU）。
        for name in batch:
            batch[name] = batch[name].to(device)

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        if step % 50 == 0 or step == len(train_loader):
            print(f"Epoch {epoch + 1}, step {step}, average loss = {total_loss / step:.4f}")

    training_losses.append(total_loss / len(train_loader))

model.save_pretrained(OUTPUT_DIR / "finetuned_bert")
tokenizer.save_pretrained(OUTPUT_DIR / "finetuned_bert")



Epoch 1, step 50, average loss = 1.1015
Epoch 1, step 100, average loss = 1.0815
Epoch 1, step 150, average loss = 1.0459
Epoch 1, step 200, average loss = 1.0420
Epoch 1, step 250, average loss = 1.0342
Epoch 1, step 300, average loss = 1.0372
Epoch 1, step 350, average loss = 1.0325
Epoch 1, step 375, average loss = 1.0327


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/nlp-hw3/output/finetuned_bert/tokenizer_config.json',
 '/content/drive/MyDrive/nlp-hw3/output/finetuned_bert/tokenizer.json')

### Prediction

`model.eval()` 關閉訓練模式，`torch.no_grad()` 表示不計算梯度。
`argmax` 選出分數最大的類別編號，再轉回標籤文字。



In [ ]:
def predict_labels(data_loader):
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in data_loader:
            for name in batch:
                batch[name] = batch[name].to(device)
            logits = model(**batch).logits
            predicted_ids = logits.argmax(dim=1).cpu().tolist()
            for label_id in predicted_ids:
                predictions.append(labels[label_id])
    return predictions

dev_predictions = predict_labels(dev_loader)
test_predictions = predict_labels(test_loader)
print("前 5 筆測試預測：", test_predictions[:5])



前 5 筆測試預測： ['NOT ENOUGH INFO', 'supports', 'supports', 'NOT ENOUGH INFO', 'supports']


### Classification Accuracy

只比對分類標籤，不要求證據正確；所有題目（包含 NEI）都進入分母。
因此可以單獨觀察 BERT 分類器的表現，不會與檢索分數混在一起。



In [ ]:
label_hits = 0
for prediction, gold_label in zip(dev_predictions, dev_df["label"]):
    if prediction.upper() == gold_label.upper():
        label_hits += 1

classification_accuracy = label_hits / len(dev_df)
print(f"Classification Accuracy: {classification_accuracy:.4f}")



Classification Accuracy: 0.3433


## 5. Validation：分開觀察三階段的表現

下列摘要會印出實際樣本數、Recall、Accuracy、候選頁面涵蓋率，以及以「前 5 句、標籤正確且證據組完整」計算的本地 strict score 估計值。實際 Kaggle 分數仍以競賽評分器為準。

執行後請將 Q1～Q3 的真實數字填入最下方 Report；`metrics.json` 也會存在輸出資料夾。Test 沒有標準答案，不用它計算這些分數。


In [ ]:
strict_hits = 0
for index, row in enumerate(dev_df.itertuples()):
    correct_label = dev_predictions[index].upper() == str(row.label).upper()
    if str(row.label).upper() == 'NOT ENOUGH INFO':
        strict_hits += correct_label
    elif correct_label and evidence_group_hit(gold_groups(row.evidence), dev_evidence[index][:5]):
        strict_hits += 1

metrics = {
    'train_samples': len(train_df),
    'dev_samples': len(dev_df),
    'verifiable_dev_samples': document_count,
    'missing_valid_gold_groups': missing_gold,
    'document_top_k': DOCUMENT_TOP_K,
    'document_recall': document_recall,
    'document_hits': document_hits,
    'document_candidate_coverage': document_coverage,
    'evidence_top_k': EVIDENCE_TOP_K,
    'evidence_recall': evidence_recall,
    'evidence_hits': evidence_hits,
    'classification_accuracy': classification_accuracy,
    'local_strict_score': strict_hits / len(dev_df),
    'retrieval_parameters': retrieval_parameters,
    'classification_parameters': classification_parameters,
    'model_name': MODEL_NAME,
    'seed': SEED,
}
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('\n請把下列實測結果填進 Report：')
print(f"Q1: dev {len(dev_df)} 筆，可驗證 {document_count} 筆；Recall@{DOCUMENT_TOP_K} = {document_recall:.4f} ({document_hits}/{document_count})。")
print(f"Q2: Recall@{EVIDENCE_TOP_K} = {evidence_recall:.4f} ({evidence_hits}/{document_count})。")
print(f"Q3: Accuracy = {classification_accuracy:.4f}；分類模型參數量 = {classification_parameters:,}。")
print(f"Q4: 候選頁面涵蓋率 = {document_coverage:.4f}；本地 strict score = {metrics['local_strict_score']:.4f}。")


{
  "train_samples": 3000,
  "dev_samples": 300,
  "verifiable_dev_samples": 201,
  "missing_valid_gold_groups": 0,
  "document_top_k": 5,
  "document_recall": 0.014925373134328358,
  "document_hits": 3,
  "document_candidate_coverage": 0.014925373134328358,
  "evidence_top_k": 5,
  "evidence_recall": 0.014925373134328358,
  "evidence_hits": 3,
  "classification_accuracy": 0.3433333333333333,
  "local_strict_score": 0.26666666666666666,
  "retrieval_parameters": 102267648,
  "classification_parameters": 102269955,
  "model_name": "google-bert/bert-base-chinese",
  "seed": 2026
}

請把下列實測結果填進 Report：
Q1: dev 300 筆，可驗證 201 筆；Recall@5 = 0.0149 (3/201)。
Q2: Recall@5 = 0.0149 (3/201)。
Q3: Accuracy = 0.3433；分類模型參數量 = 102,269,955。
Q4: 候選頁面涵蓋率 = 0.0149；本地 strict score = 0.2667。


## 6. 產生 submission.csv

提交檔需要 `id`、`predicted_label`、`predicted_evidence` 三欄。
證據使用 `json.dumps()` 轉成合法 JSON 字串，NEI 的證據填 `[]`。
保持測試資料的 ID 與順序，再從 Kaggle Notebook 輸出提交這份 CSV。



In [ ]:
evidence_strings = []
for label, evidence in zip(test_predictions, test_evidence):
    if label == 'NOT ENOUGH INFO':
        evidence = []
    evidence_strings.append(json.dumps(evidence, ensure_ascii=False))

submission = pd.DataFrame({
    'id': test_df['id'],
    'predicted_label': test_predictions,
    'predicted_evidence': evidence_strings,
})
assert len(submission) == len(test_df)
assert submission['id'].tolist() == test_df['id'].tolist()
assert submission['predicted_label'].isin(labels).all()
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False, encoding='utf-8')
print('輸出筆數：', len(submission))
print('輸出位置：', OUTPUT_DIR / 'submission.csv')
submission.head()


輸出筆數： 6000
輸出位置： /content/drive/MyDrive/nlp-hw3/output/submission.csv


,id,predicted_label,predicted_evidence
0,29579,NOT ENOUGH INFO,[]
1,3895,supports,"[[""江西省"", 1], [""納木錯"", 1], [""江西省"", 0], [""九江市"", 3..."
2,24522,supports,"[[""伍德羅·威爾遜"", 5], [""威廉·亨利·哈里森"", 0], [""亞伯拉罕·林肯"",..."
3,11060,NOT ENOUGH INFO,[]
4,23607,supports,"[[""胡德號戰列巡洋艦"", 2], [""胡德號戰列巡洋艦"", 8], [""俾斯麥號戰艦"", ..."


# Report

Your Kaggle username: zhi816

Your public leaderboard score: 待提交 Kaggle 後填入實測分數。

Your private leaderboard score: 待 2026/11/23 12:00 公布後填入實測分數。


1. Describe your method for retrieving Wikipedia documents (5%) and report Recall@N on development set (5%).

Answer: 我先從 CFEVER 全部公開訓練資料的標準證據取出頁面名稱，再從 Wikipedia 讀入這些頁面，作為這次的候選文件。每頁將標題與內文開頭送進 `google-bert/bert-base-chinese`；claim 也用同一個凍結的 BERT 編碼。我以 attention mask 排除 padding，將 token 向量做 mean pooling 和 L2 normalization，再用內積計算 cosine similarity，選出前 5 頁。候選庫不含訓練標註以外的頁面，因此這是一個有涵蓋率上限的基線。Development set 文件 Recall@5：待執行後依第 5 節輸出填入數值、命中題數及可驗證題數；預設抽取 300 筆 dev，實際筆數請以輸出為準。

2. Describe your method for retrieving evidence (5%) and report Recall@N on development set  (5%).

Answer: 我只從第一階段取回的頁面挑選證據，將這些頁面的非空句子以同一個凍結 BERT 轉成向量，再依 claim 與句子的 cosine similarity 選前 5 句。句子保留 Wikipedia 原始頁名和句號，沒有只限定在訓練集標註過的句子。Development set 證據 Recall@5：待執行後依第 5 節輸出填入數值、命中題數及可驗證題數；必須完整找到任一組標準證據才算命中。

3. Describe your classification method (5%) and report Accuracy on development set  (5%).

Answer: 我將 claim 和檢索到的證據串成一對文字，透過 tokenizer 形成 `[CLS] claim [SEP] evidence [SEP]`，再使用 `google-bert/bert-base-chinese` 加上三分類層，預測 supports、refutes 或 NOT ENOUGH INFO。分類器與前面凍結的檢索模型是分開的模型；訓練採 AdamW、learning rate 2e-5、batch size 8、1 epoch、最大長度 192，預設使用 3,000 筆訓練樣本。模型參數量由程式印出並檢查小於 1B。Development set Accuracy：待執行後依第 5 節輸出填入數值與實際 dev 筆數。這個 Accuracy 使用實際檢索到的證據作為輸入，只檢查分類標籤。

4. … Anything that can strengthen your report. (10%)

Answer: 我另外觀察候選頁面涵蓋率與本地 strict score：前者檢查候選頁面是否包含任一完整標準證據組所需頁面；後者對可驗證題目同時要求標籤正確、前 5 句證據完整，NEI 則只看標籤。實際數值待執行後由第 5 節輸出補入，競賽成績仍以 Kaggle 評分為準。我認為這個方法的限制是候選頁面只來自訓練標註、BERT 沒有針對檢索微調，而且分類器可能看到錯誤的檢索句子；後續可以比較候選頁面範圍、top-k 與不同訓練筆數，並用相同 dev split 觀察變化。以上是分析與後續方向，並非已完成的消融實驗。

5. Generative AI Usage Statement

Answer: 我使用 ChatGPT / OpenAI Codex 協助理解題目、改寫三階段程式、整理報告草稿和學習筆記，以及檢查流程。主要提示為「依作業規範改寫 notebook、回答報告題目，並整理可能會問的問題和核心程式學習指南」。我會自行閱讀、執行與核對程式和實驗結果，並依實際使用狀況修正這份聲明；程式開頭也有註記 AI 使用及參考來源。
